In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip -q install -U transformers datasets peft accelerate scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 w

In [1]:
%%writefile train_fedavg_bertbase_full_imdb.py
import os
import gc
import json
import time
import glob
import shutil
import random
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)

# =========================
# CONFIG
# =========================
@dataclass
class Config:
    # ===== Model / Data =====
    model_name: str = "bert-base-uncased"
    dataset_name: str = "imdb"
    setting: str = "FedAvg + BERT-base (full fine-tuning)"
    num_labels: int = 2
    text_column: str = "text"
    label_column: str = "label"
    seed: int = 42

    # ===== Output =====
    drive_root: str = "/content/drive/MyDrive/fedavg_bertbase_full_imdb"
    experiment_name: str = "fedavg_bertbase_full_imdb"

    # ===== Tokenization / Data =====
    max_length: int = 256
    validation_ratio: float = 0.1

    # ===== Federated Learning =====
    num_clients: int = 5
    local_epochs: int = 1
    local_batch_size: int = 8
    eval_batch_size: int = 32
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0

    # ===== Partition =====
    partition_type: str = "noniid"   # "iid" or "noniid"
    dirichlet_alpha: float = 0.5

    # ===== Early stopping =====
    monitor: str = "accuracy"
    greater_is_better: bool = True
    early_stopping_patience: int = 3
    min_delta: float = 0.0

    # ===== Runtime =====
    fp16: bool = False
    num_workers: int = 0
    resume: bool = True
    save_latest_k: int = 2
    evaluate_each_client: bool = True
    max_rounds: int = 1000

    @property
    def exp_dir(self):
        return os.path.join(self.drive_root, self.experiment_name)

    @property
    def ckpt_dir(self):
        return os.path.join(self.exp_dir, "checkpoints")

    @property
    def best_model_dir(self):
        return os.path.join(self.exp_dir, "best_model")

    @property
    def final_model_dir(self):
        return os.path.join(self.exp_dir, "final_model")

    @property
    def history_json_path(self):
        return os.path.join(self.exp_dir, "history.json")

    @property
    def history_csv_path(self):
        return os.path.join(self.exp_dir, "history.csv")

    @property
    def client_history_json_path(self):
        return os.path.join(self.exp_dir, "client_history.json")

    @property
    def client_history_csv_path(self):
        return os.path.join(self.exp_dir, "client_history.csv")

    @property
    def config_path(self):
        return os.path.join(self.exp_dir, "config.json")

    @property
    def summary_path(self):
        return os.path.join(self.exp_dir, "run_summary.json")


# =========================
# UTILS
# =========================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def ensure_dirs(cfg: Config):
    os.makedirs(cfg.drive_root, exist_ok=True)
    os.makedirs(cfg.exp_dir, exist_ok=True)
    os.makedirs(cfg.ckpt_dir, exist_ok=True)
    os.makedirs(cfg.best_model_dir, exist_ok=True)
    os.makedirs(cfg.final_model_dir, exist_ok=True)

def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def save_history_csv(path, history):
    if len(history) == 0:
        return
    pd.DataFrame(history).to_csv(path, index=False, encoding="utf-8")

def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

def metric_improved(current, best, greater_is_better=True, min_delta=0.0):
    if greater_is_better:
        return current > (best + min_delta)
    return current < (best - min_delta)

def tokenize_function(examples, tokenizer, text_column, max_length):
    return tokenizer(
        examples[text_column],
        truncation=True,
        max_length=max_length,
    )

def dirichlet_noniid_partition(labels, num_clients=5, alpha=0.5, seed=42, min_size=10):
    rng = np.random.default_rng(seed)
    labels = np.array(labels)
    unique_classes = np.unique(labels)

    while True:
        client_indices = [[] for _ in range(num_clients)]

        for c in unique_classes:
            idx = np.where(labels == c)[0]
            rng.shuffle(idx)

            proportions = rng.dirichlet(np.repeat(alpha, num_clients))
            split_points = (np.cumsum(proportions) * len(idx)).astype(int)[:-1]
            splits = np.split(idx, split_points)

            for client_id, split in enumerate(splits):
                client_indices[client_id].extend(split.tolist())

        for client_id in range(num_clients):
            rng.shuffle(client_indices[client_id])

        sizes = [len(x) for x in client_indices]
        if min(sizes) >= min_size:
            return client_indices

def iid_partition(num_samples, num_clients=5, seed=42):
    rng = np.random.default_rng(seed)
    indices = np.arange(num_samples)
    rng.shuffle(indices)
    splits = np.array_split(indices, num_clients)
    return [split.tolist() for split in splits]

def create_model(cfg: Config):
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=cfg.num_labels
    )
    return model

def get_model_state_dict_cpu(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def load_model_state_dict(model, state_dict):
    model.load_state_dict(state_dict, strict=True)

def average_model_states(client_states, client_weights):
    if len(client_states) == 0:
        raise ValueError("client_states is empty.")

    avg_state = {}
    keys = client_states[0].keys()
    total_weight = float(sum(client_weights))
    if total_weight <= 0:
        raise ValueError("Total client weights must be > 0.")

    norm_weights = [w / total_weight for w in client_weights]

    for k in keys:
        avg_tensor = None
        for state, w in zip(client_states, norm_weights):
            tensor = state[k].float() * w
            if avg_tensor is None:
                avg_tensor = tensor
            else:
                avg_tensor += tensor
        avg_state[k] = avg_tensor
    return avg_state

def compute_comm_cost_mb(model_state_dict, num_clients):
    total_bytes_one_way = 0
    for _, tensor in model_state_dict.items():
        total_bytes_one_way += tensor.numel() * tensor.element_size()
    total_bytes_round = total_bytes_one_way * num_clients * 2  # broadcast + upload
    return total_bytes_round / (1024 ** 2)

def list_checkpoints(ckpt_dir: str):
    ckpts = []
    if not os.path.exists(ckpt_dir):
        return ckpts

    for path in glob.glob(os.path.join(ckpt_dir, "checkpoint-round-*")):
        try:
            round_num = int(path.split("-")[-1])
            ckpts.append((round_num, path))
        except Exception:
            pass

    ckpts.sort(key=lambda x: x[0])
    return ckpts

def latest_checkpoint_path(ckpt_dir: str):
    ckpts = list_checkpoints(ckpt_dir)
    return ckpts[-1][1] if ckpts else None

def prune_old_checkpoints(ckpt_dir: str, keep_latest_k: int = 2):
    ckpts = list_checkpoints(ckpt_dir)
    while len(ckpts) > keep_latest_k:
        _, old_path = ckpts.pop(0)
        try:
            shutil.rmtree(old_path)
        except Exception:
            pass

def clear_dir_keep_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

def save_training_artifacts(cfg: Config, save_dir: str, model_state_dict, tokenizer):
    clear_dir_keep_dir(save_dir)

    torch.save(model_state_dict, os.path.join(save_dir, "global_model_state.pt"))

    save_model = create_model(cfg)
    load_model_state_dict(save_model, model_state_dict)
    save_model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

    save_json(os.path.join(save_dir, "model_info.json"), {
        "model_name": cfg.model_name,
        "dataset_name": cfg.dataset_name,
        "setting": cfg.setting,
        "num_labels": cfg.num_labels,
    })

    del save_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def save_checkpoint(
    cfg,
    round_idx,
    global_model_state,
    tokenizer,
    best_metric,
    patience_counter,
    history,
    client_history
):
    ckpt_path = os.path.join(cfg.ckpt_dir, f"checkpoint-round-{round_idx}")
    os.makedirs(ckpt_path, exist_ok=True)

    torch.save(global_model_state, os.path.join(ckpt_path, "global_model_state.pt"))

    ckpt_model = create_model(cfg)
    load_model_state_dict(ckpt_model, global_model_state)
    ckpt_model.save_pretrained(ckpt_path)
    tokenizer.save_pretrained(ckpt_path)

    state = {
        "round": round_idx,
        "best_metric": best_metric,
        "patience_counter": patience_counter,
        "history": history,
        "client_history": client_history,
        "config": asdict(cfg),
    }
    torch.save(state, os.path.join(ckpt_path, "training_state.pt"))
    save_json(os.path.join(ckpt_path, "config.json"), asdict(cfg))

    with open(os.path.join(cfg.ckpt_dir, "latest_checkpoint.txt"), "w", encoding="utf-8") as f:
        f.write(ckpt_path)

    prune_old_checkpoints(cfg.ckpt_dir, cfg.save_latest_k)

    del ckpt_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return ckpt_path

def load_latest_checkpoint(cfg, device="cpu"):
    ckpt_path = latest_checkpoint_path(cfg.ckpt_dir)
    if ckpt_path is None or not cfg.resume:
        return None, 0, None, 0, [], []

    model_state_path = os.path.join(ckpt_path, "global_model_state.pt")
    state_path = os.path.join(ckpt_path, "training_state.pt")
    config_path = os.path.join(ckpt_path, "config.json")

    if not (os.path.exists(model_state_path) and os.path.exists(state_path) and os.path.exists(config_path)):
        print("Checkpoint files missing. Start from scratch.")
        return None, 0, None, 0, [], []

    print(f"Resuming from checkpoint: {ckpt_path}")

    server_model = create_model(cfg).to(device)
    model_state = torch.load(model_state_path, map_location="cpu")
    load_model_state_dict(server_model, model_state)

    state = torch.load(state_path, map_location=device)

    return (
        server_model,
        state["round"],
        state.get("best_metric", None),
        state.get("patience_counter", 0),
        state.get("history", []),
        state.get("client_history", []),
    )

@torch.no_grad()
def evaluate(model, loader, device, use_amp=False, split_name="validation"):
    model.eval()

    total_loss = 0.0
    total_steps = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(loader, desc=f"Evaluating {split_name}", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}

        if use_amp:
            with torch.cuda.amp.autocast(enabled=True):
                outputs = model(**batch)
                loss = outputs.loss
                logits = outputs.logits
        else:
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

        preds = torch.argmax(logits, dim=-1)

        total_loss += loss.item()
        total_steps += 1
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(batch["labels"].detach().cpu().numpy().tolist())

    return {
        "eval_loss": float(total_loss / max(total_steps, 1)),
        "accuracy": float(accuracy_score(all_labels, all_preds)),
        "macro_f1": float(f1_score(all_labels, all_preds, average="macro")),
    }

def prepare_data(cfg: Config, tokenizer):
    ds = load_dataset(cfg.dataset_name)

    # IMDB có train/test chuẩn.
    # Ta tách validation từ train để phục vụ early stopping và logging.
    split_ds = ds["train"].train_test_split(
        test_size=cfg.validation_ratio,
        seed=cfg.seed,
        shuffle=True
    )

    train_ds = split_ds["train"]
    val_ds = split_ds["test"]

    remove_columns_train = [cfg.text_column]
    remove_columns_val = [cfg.text_column]

    tokenized_train = train_ds.map(
        lambda x: tokenize_function(x, tokenizer, cfg.text_column, cfg.max_length),
        batched=True,
        remove_columns=remove_columns_train,
    )
    tokenized_val = val_ds.map(
        lambda x: tokenize_function(x, tokenizer, cfg.text_column, cfg.max_length),
        batched=True,
        remove_columns=remove_columns_val,
    )

    tokenized_train = tokenized_train.rename_column(cfg.label_column, "labels")
    tokenized_val = tokenized_val.rename_column(cfg.label_column, "labels")

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    labels = tokenized_train["labels"]

    if cfg.partition_type.lower() == "noniid":
        client_splits = dirichlet_noniid_partition(
            labels=labels,
            num_clients=cfg.num_clients,
            alpha=cfg.dirichlet_alpha,
            seed=cfg.seed,
            min_size=10
        )
    else:
        client_splits = iid_partition(
            num_samples=len(tokenized_train),
            num_clients=cfg.num_clients,
            seed=cfg.seed
        )

    client_loaders = []
    client_num_samples = []

    for client_id in range(cfg.num_clients):
        subset = Subset(tokenized_train, client_splits[client_id])
        loader = DataLoader(
            subset,
            batch_size=cfg.local_batch_size,
            shuffle=True,
            collate_fn=collator,
            num_workers=cfg.num_workers,
            pin_memory=torch.cuda.is_available(),
        )
        client_loaders.append(loader)
        client_num_samples.append(len(client_splits[client_id]))

    val_loader = DataLoader(
        tokenized_val,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        collate_fn=collator,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return client_loaders, client_num_samples, val_loader

def train_one_client(cfg, global_model_state, client_loader, client_id, num_samples, device, eval_loader):
    # ===== server broadcast model =====
    client_model = create_model(cfg).to(device)
    load_model_state_dict(client_model, global_model_state)

    total_params, trainable_params = count_parameters(client_model)

    use_amp = cfg.fp16 and device == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    optimizer = torch.optim.AdamW(
        client_model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay
    )

    client_model.train()
    client_start = time.time()
    running_loss = 0.0
    total_steps = 0

    # ===== client local training =====
    for local_epoch in range(cfg.local_epochs):
        pbar = tqdm(
            client_loader,
            desc=f"Client {client_id} epoch {local_epoch + 1}/{cfg.local_epochs}",
            leave=False
        )

        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with torch.cuda.amp.autocast(enabled=True):
                    outputs = client_model(**batch)
                    loss = outputs.loss

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(client_model.parameters(), cfg.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = client_model(**batch)
                loss = outputs.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(client_model.parameters(), cfg.max_grad_norm)
                optimizer.step()

            running_loss += loss.item()
            total_steps += 1

            if total_steps % 20 == 0:
                pbar.set_postfix(loss=f"{loss.item():.4f}")

    client_train_time = time.time() - client_start
    avg_train_loss = float(running_loss / max(total_steps, 1))

    # ===== client evaluation =====
    if cfg.evaluate_each_client:
        client_eval = evaluate(
            client_model,
            eval_loader,
            device=device,
            use_amp=use_amp,
            split_name=f"client_{client_id}_validation"
        )
    else:
        client_eval = {
            "eval_loss": None,
            "accuracy": None,
            "macro_f1": None,
        }

    # ===== client upload =====
    client_model_state = get_model_state_dict_cpu(client_model)

    del client_model
    del optimizer
    del scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "client_state": client_model_state,
        "num_samples": int(num_samples),
        "train_time": float(client_train_time),
        "train_loss": float(avg_train_loss),
        "eval_loss": None if client_eval["eval_loss"] is None else float(client_eval["eval_loss"]),
        "accuracy": None if client_eval["accuracy"] is None else float(client_eval["accuracy"]),
        "macro_f1": None if client_eval["macro_f1"] is None else float(client_eval["macro_f1"]),
        "trainable_params": int(trainable_params),
        "total_params": int(total_params),
    }

def verify_required_outputs(cfg: Config):
    required_paths = [
        cfg.history_json_path,
        cfg.history_csv_path,
        cfg.client_history_json_path,
        cfg.client_history_csv_path,
        cfg.summary_path,
        cfg.config_path,
        os.path.join(cfg.best_model_dir, "config.json"),
        os.path.join(cfg.best_model_dir, "global_model_state.pt"),
        os.path.join(cfg.final_model_dir, "config.json"),
        os.path.join(cfg.final_model_dir, "global_model_state.pt"),
    ]

    missing = [p for p in required_paths if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError("Missing required output files:\\n" + "\\n".join(missing))

def verify_log_columns(history, client_history):
    required_server_cols = [
        "round", "accuracy", "macro_f1", "eval_loss", "round_time",
        "client_train_time", "communication_cost_MB", "trainable_params", "total_params",
        "model_name", "dataset_name", "setting", "num_clients", "local_epochs",
        "partition_type", "best_metric_so_far", "patience_counter", "is_new_best"
    ]
    required_client_cols = required_server_cols + ["client_id", "num_samples", "train_loss"]

    if len(history) > 0:
        for col in required_server_cols:
            if col not in history[-1]:
                raise ValueError(f"Missing server log column: {col}")

    if len(client_history) > 0:
        for col in required_client_cols:
            if col not in client_history[-1]:
                raise ValueError(f"Missing client log column: {col}")

# =========================
# TRAIN
# =========================
def train():
    cfg = Config()
    set_seed(cfg.seed)
    ensure_dirs(cfg)
    save_json(cfg.config_path, asdict(cfg))

    device = get_device()
    print("=" * 100)
    print("Device:", device)
    print("Experiment dir:", cfg.exp_dir)
    print("=" * 100)

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)

    client_loaders, client_num_samples, val_loader = prepare_data(cfg, tokenizer)

    resumed_model, start_round, best_metric, patience_counter, history, client_history = load_latest_checkpoint(
        cfg, device=device
    )

    if resumed_model is not None:
        server_model = resumed_model
    else:
        server_model = create_model(cfg).to(device)
        start_round = 0
        best_metric = None
        patience_counter = 0
        history = []
        client_history = []

    total_params, trainable_params = count_parameters(server_model)

    if best_metric is None:
        best_metric = -1e18 if cfg.greater_is_better else 1e18

    print(f"Start from round {start_round + 1}")
    print(f"Best {cfg.monitor} so far: {best_metric:.6f}")
    print(f"Patience counter: {patience_counter}")
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")

    stopped_early = False
    round_idx = start_round + 1

    while round_idx <= cfg.max_rounds:
        round_start = time.time()

        # ===== server broadcast model =====
        global_model_state = get_model_state_dict_cpu(server_model)

        client_states = []
        client_weights = []
        client_times = []
        round_client_logs = []

        # ===== client local training =====
        for client_id in range(cfg.num_clients):
            print(f"\\n[Round {round_idx}] Training client {client_id + 1}/{cfg.num_clients}")

            client_result = train_one_client(
                cfg=cfg,
                global_model_state=global_model_state,
                client_loader=client_loaders[client_id],
                client_id=client_id,
                num_samples=client_num_samples[client_id],
                device=device,
                eval_loader=val_loader
            )

            client_states.append(client_result["client_state"])
            client_weights.append(client_result["num_samples"])
            client_times.append(client_result["train_time"])

            client_log = {
                "round": int(round_idx),
                "accuracy": client_result["accuracy"],
                "macro_f1": client_result["macro_f1"],
                "eval_loss": client_result["eval_loss"],
                "round_time": None,
                "client_train_time": float(client_result["train_time"]),
                "communication_cost_MB": None,
                "trainable_params": int(client_result["trainable_params"]),
                "total_params": int(client_result["total_params"]),
                "model_name": cfg.model_name,
                "dataset_name": cfg.dataset_name,
                "setting": cfg.setting,
                "num_clients": int(cfg.num_clients),
                "local_epochs": int(cfg.local_epochs),
                "partition_type": cfg.partition_type,
                "best_metric_so_far": None,
                "patience_counter": None,
                "is_new_best": None,
                "client_id": int(client_id),
                "num_samples": int(client_result["num_samples"]),
                "train_loss": float(client_result["train_loss"]),
            }

            round_client_logs.append(client_log)

        # ===== server aggregation =====
        aggregated_model_state = average_model_states(client_states, client_weights)
        load_model_state_dict(server_model, aggregated_model_state)

        # ===== server validation =====
        use_amp = cfg.fp16 and device == "cuda"
        val_metrics = evaluate(
            server_model,
            val_loader,
            device=device,
            use_amp=use_amp,
            split_name="validation"
        )

        round_time = time.time() - round_start
        client_train_time_total = float(sum(client_times))
        communication_cost_MB = float(compute_comm_cost_mb(global_model_state, cfg.num_clients))

        current_metric = val_metrics[cfg.monitor]
        is_new_best = metric_improved(
            current=current_metric,
            best=best_metric,
            greater_is_better=cfg.greater_is_better,
            min_delta=cfg.min_delta
        )

        if is_new_best:
            best_metric = current_metric
            patience_counter = 0
            save_training_artifacts(
                cfg=cfg,
                save_dir=cfg.best_model_dir,
                model_state_dict=aggregated_model_state,
                tokenizer=tokenizer
            )
        else:
            patience_counter += 1

        round_result = {
            "round": int(round_idx),
            "accuracy": float(val_metrics["accuracy"]),
            "macro_f1": float(val_metrics["macro_f1"]),
            "eval_loss": float(val_metrics["eval_loss"]),
            "round_time": float(round_time),
            "client_train_time": float(client_train_time_total),
            "communication_cost_MB": float(communication_cost_MB),
            "trainable_params": int(trainable_params),
            "total_params": int(total_params),
            "model_name": cfg.model_name,
            "dataset_name": cfg.dataset_name,
            "setting": cfg.setting,
            "num_clients": int(cfg.num_clients),
            "local_epochs": int(cfg.local_epochs),
            "partition_type": cfg.partition_type,
            "best_metric_so_far": float(best_metric),
            "patience_counter": int(patience_counter),
            "is_new_best": bool(is_new_best),
        }

        history.append(round_result)

        for client_log in round_client_logs:
            client_log["round_time"] = float(round_time)
            client_log["communication_cost_MB"] = float(communication_cost_MB)
            client_log["best_metric_so_far"] = float(best_metric)
            client_log["patience_counter"] = int(patience_counter)
            client_log["is_new_best"] = bool(is_new_best)
            client_history.append(client_log)

        verify_log_columns(history, client_history)

        save_json(cfg.history_json_path, history)
        save_history_csv(cfg.history_csv_path, history)

        save_json(cfg.client_history_json_path, client_history)
        save_history_csv(cfg.client_history_csv_path, client_history)

        ckpt_path = save_checkpoint(
            cfg=cfg,
            round_idx=round_idx,
            global_model_state=aggregated_model_state,
            tokenizer=tokenizer,
            best_metric=best_metric,
            patience_counter=patience_counter,
            history=history,
            client_history=client_history
        )

        print("-" * 100)
        print("SERVER ROUND LOG:")
        print(json.dumps(round_result, indent=2))
        print("CLIENT ROUND LOGS:")
        print(json.dumps(round_client_logs, indent=2))
        print("Checkpoint saved:", ckpt_path)
        print("-" * 100)

        if patience_counter >= cfg.early_stopping_patience:
            print(f"Early stopping triggered: no improvement in {cfg.early_stopping_patience} consecutive rounds.")
            stopped_early = True
            break

        round_idx += 1

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_model_state = get_model_state_dict_cpu(server_model)
    save_training_artifacts(
        cfg=cfg,
        save_dir=cfg.final_model_dir,
        model_state_dict=final_model_state,
        tokenizer=tokenizer
    )

    run_summary = {
        "model_name": cfg.model_name,
        "dataset_name": cfg.dataset_name,
        "setting": cfg.setting,
        "monitor_metric": cfg.monitor,
        "best_metric_so_far": float(best_metric),
        "num_rounds_completed": int(len(history)),
        "num_client_logs": int(len(client_history)),
        "stopped_early": bool(stopped_early),
        "best_model_dir": cfg.best_model_dir,
        "final_model_dir": cfg.final_model_dir,
        "latest_checkpoint": latest_checkpoint_path(cfg.ckpt_dir),
        "history_json": cfg.history_json_path,
        "history_csv": cfg.history_csv_path,
        "client_history_json": cfg.client_history_json_path,
        "client_history_csv": cfg.client_history_csv_path,
    }
    save_json(cfg.summary_path, run_summary)

    verify_required_outputs(cfg)
    verify_log_columns(history, client_history)

    print("=" * 100)
    print("OUTPUT DIRECTORY          :", cfg.exp_dir)
    print("Best model dir            :", cfg.best_model_dir)
    print("Final model dir           :", cfg.final_model_dir)
    print("Checkpoints dir           :", cfg.ckpt_dir)
    print("Server history JSON       :", cfg.history_json_path)
    print("Server history CSV        :", cfg.history_csv_path)
    print("Client history JSON       :", cfg.client_history_json_path)
    print("Client history CSV        :", cfg.client_history_csv_path)
    print("Run summary               :", cfg.summary_path)
    print("=" * 100)
    print("Done.")

if __name__ == "__main__":
    train()

Writing train_fedavg_bertbase_full_imdb.py


In [2]:
!python train_fedavg_bertbase_full_imdb.py

Device: cuda
Experiment dir: /content/drive/MyDrive/fedavg_bertbase_full_imdb/fedavg_bertbase_full_imdb
config.json: 100% 570/570 [00:00<00:00, 2.54MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 214kB/s]
vocab.txt: 232kB [00:00, 11.0MB/s]
tokenizer.json: 466kB [00:00, 3.82MB/s]
README.md: 7.81kB [00:00, 21.3MB/s]
plain_text/train-00000-of-00001.parquet: 100% 21.0M/21.0M [00:00<00:00, 29.6MB/s]
plain_text/test-00000-of-00001.parquet: 100% 20.5M/20.5M [00:00<00:00, 33.2MB/s]
plain_text/unsupervised-00000-of-00001.p(…): 100% 42.0M/42.0M [00:00<00:00, 68.7MB/s]
Generating train split: 100% 25000/25000 [00:00<00:00, 118158.20 examples/s]
Generating test split: 100% 25000/25000 [00:00<00:00, 187489.12 examples/s]
Generating unsupervised split: 100% 50000/50000 [00:00<00:00, 176430.37 examples/s]
Map: 100% 22500/22500 [00:19<00:00, 1152.68 examples/s]
Map: 100% 2500/2500 [00:02<00:00, 947.51 examples/s]
model.safetensors: 100% 440M/440M [00:02<00:00, 219MB/s]
Loading weights: 100% 